In [1]:
import json
import os
from datetime import datetime, timedelta
from crewai import Agent, Task, Crew, Process
from crewai_tools import CSVSearchTool
from dotenv import load_dotenv, find_dotenv
from langchain.tools import Tool
from langchain_community.tools import DuckDuckGoSearchResults


In [2]:
from langchain_openai import ChatOpenAI

load_dotenv( find_dotenv(), override=True )

# Verificar qual chave está sendo usada (mostra só os últimos 4 caracteres)
api_key = os.environ.get("OPENAI_API_KEY", "NÃO ENCONTRADA")
print(f"Chave carregada termina em: ...{api_key[-4:]}")


llm = ChatOpenAI(model="gpt-3.5-turbo-0125")

Chave carregada termina em: ...js8A


In [3]:
csv_imoveis = CSVSearchTool(csv="files/imoveis.csv")

In [4]:
# Agente Corretor de Imóveis
corretor_imoveis = Agent(
    role="Corretor Imóveis",
    goal="Obtenha as preferências do cliente e busque imóveis compatíveis no banco de dados",
    backstory="Especialista no mercado imbiliário, encontra as melhores opções baseadas no perfil do cliente",
    verbose=True,
    max_iter=5,
    tools=[csv_imoveis],
    memory=True
)

In [5]:
# Tarefa -  Buscar Imóveis
buscar_imoveis = Task(
    description="Pesquise imóveis na região deseja pelo cliente considerando faixa de preço e tipo de imóvel",
    expected_output="Lista de imóveis disponíveis com detalhes sobre localização, preço e características",
    agent=corretor_imoveis
)

In [6]:
from crewai.tools import BaseTool

def obter_precos_imoveis(cidade: str = "geral"):
    precos = {
        "São Paulo": {"tendencia": "aumento", "percentual": 5.2},
        "Rio de Janeiro": {"tendencia": "estavel", "percentual": 0.0},
        "Belo Horizonte": {"tendencia": "queda", "percentual": -3.1},
        "geral": {"tendencia": "aumento", "percentual": 4.0},
    }
    return precos.get(cidade, precos["geral"])

class TendenciaPrecosImoveisTool(BaseTool):
    name: str = "Analisador de Preços Imbiliários"
    description: str = "Obtém tendências de preços de imóveis com base na cidade especificaada"

    def _run(self, cidade: str) -> dict:
        """
        Execita a análise re preço imobiliários e retorna a tendência com base na cidade
        """
        try:
            return obter_precos_imoveis(cidade)
        except Exception as e:
            return {"erro": f"Erro ao obter tendências de preços {str(e)}"}

In [7]:
analista_mercado = Agent(
    role="Analista de Mercado Imobiliário",
    goal="Analista tendências de preços e ajuda a prever a valorização ou desvalorização dos imóveis da cidade {cidade}",
    backstory="Experiente no setor, usa dados históricos para prever preços futuros.",
    verbose=True,
    max_iter=5,
    allow_delegation=False,
    memory=True
)

In [8]:
obter_tendencias = Task(
    description="""
    Analise o histórico de preços de imóveis na cidade {cidade} e forneça insights sobre
    valorização ou desvalorização. Considere o tipo de imóvel {tipo_imovel} e a 
    faixa de preço {faixa_preco}
    """,
    expected_output="Resumo da tendência dos preços no mercado imobiliário",
    tools=[TendenciaPrecosImoveisTool()],
    agent=analista_mercado,
    parameters=["cidade"]
)

In [9]:
analista_noticias = Agent(
    role="Analista de Notícias Imobiliários",
    goal="Busca notícias relevantes sobre o mercado imbobiliário para avaliar fatores externos.",
    backstory="Especialista em analisar notícias e tendências econômicas que afetam os preços dos imóveis",
    verbose=True,
    max_iter=5,
    memory=True
)

In [10]:
searchTool = DuckDuckGoSearchResults(backend="news", num_results=5)

In [11]:
searchTool

DuckDuckGoSearchResults(max_results=5, api_wrapper=DuckDuckGoSearchAPIWrapper(region='wt-wt', safesearch='moderate', time='y', max_results=5, backend='auto', source='text'), backend='news')

In [12]:
buscar_noticias = Task(
    description=f"Pesquise notícias recentes sobre o mercado imobiliário. Data atual: {datetime.now}",
    expected_output="Resumo das principais notíias e tendências imobiliárias",
    agent=analista_noticias,
    tool=[searchTool]
)

In [13]:
consultor_financeiro = Agent(
    role="Consultor Financeiro",
    goal="Analisa opções de financiamento imbiliário com base no perfil do cliente",
    backstory="Especialista em crédito imobiliário, ajuda clientes a escolherem as melhores opções de financiamento",
    verbose=True,
    allow_delegation=False,
    max_iter=5,
    memory=True
)

In [14]:
calcular_financinamento = Task(
    description="Analise a renda do cliente e sugira opções de financiamento viáveis.",
    expected_output="Tabela comparativa com diferentes financiamentos, taxa de juros e prazos",
    agent=consultor_financeiro
)

In [15]:
redator = Agent(
    role="Redator de Relatórios Imobiliários",
    goal="Gera um relatório completo e persuasivo com base nas análises de mercado e imóveis encontrados.",
    backstory="Especialista em comunicação, traduz dados complexos para clientes de forma clara e objetiva.",
    verbose=True,
    allow_delegation=False,
    max_iter=5,
    memory=True
)

In [16]:
gerar_relatorio = Task(
    description="Gere um relatório detalhado sobre o melhor imóvel encontrado, considerando preços, tendências e financiamento",
    expected_output="Relatório formatado com resumo do mercado, opções recomendadas e justificativas da escolha.",
    agent=redator,
    context=[buscar_imoveis, obter_tendencias, buscar_noticias, calcular_financinamento]
)

In [17]:
crew = Crew(
    agents=[corretor_imoveis, analista_mercado, analista_noticias, consultor_financeiro, redator],
    tasks=[buscar_imoveis, obter_tendencias, buscar_noticias, calcular_financinamento, gerar_relatorio],
    verbose=True,
    process=Process.hierarchical,
    full_output=True,
    share_crew=False,
    max_iter=15,
    manager_llm=llm
)

In [18]:
result = crew.kickoff(inputs={"cidade": "Rio de Janeiro",
                              "tipo_imovel": "apartamento",
                              "faixa_preco": "500000-700000"})

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  7ea7c3c7-6cfb-4c32-bba7-e023554d6ee4                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Pesquise imóveis na região deseja pelo cliente considerando faixa de preço e tipo de imóvel              │
│  ID: 79cf3d94-b903-413e-885d-b02658af9a80                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Task: Pesquise imóveis na região deseja pelo cliente considerando faixa de preço e tipo de imóvel              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_a_csvs_content                                                                                    │
│  Args: {'search_query': 'região desejada: Copacabana, faixa de preço: R$500,000 - R$700,000, tipo de imóvel:    │
│  Apartamento'}                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool search_a_csvs_content executed with result: Relevant Content:

Row 7: ID: 7 | Endereço: Rua SV de Novembro, 789 - Curitiba | Preço: 2740133 | Quartos: 4 | Banheiros: 4 | Metragem: 176 | Tipo: Sobado
Row 
Row 8: ID: 8 | Endereço: Rua das Palmeir...


...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The tool successfully provided a detailed and well-structured report on the best property found for the        │
│  client, considering prices, trends, and financing options.                                                     │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The report includes an in-depth analysis of the properties in São Paulo and Porto Alegre, comparison of        │
│  market trends, financing options, and a recommendation based on the analysis.                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Gere um relatório detalhado sobre o melhor imóvel encontrado, considerando preços, tendências e financiamento  │
│  Agent:                                                                                                         │
│  Crew Manager                                                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  7ea7c3c7-6cfb-4c32-bba7-e023554d6ee4                                                                           │
│  Final Output: The tool successfully provided a detailed and well-structured report on the best property found  │
│  for the client, considering prices, trends, and financing options.                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The report includes an in-depth analysis of the properties in São Paulo and Porto Alegre, comparison of        │
│  market trends, financing options, and a recommendation based on the analysis.                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



╭────────────────────────────────────────────────── Execution Traces ──────────────────────────────────────────────────╮
│                                                                                                                      │
│  🔍 Detailed execution traces are available!                                                                         │
│                                                                                                                      │
│  View insights including:                                                                                            │
│    • Agent decision-making process                                                                                   │
│    • Task execution flow and timing                                                                                  │
│    • Tool usage details                                                                                              │
│                              

In [19]:
print(result.raw)

The tool successfully provided a detailed and well-structured report on the best property found for the client, considering prices, trends, and financing options.

Final Answer:
The report includes an in-depth analysis of the properties in São Paulo and Porto Alegre, comparison of market trends, financing options, and a recommendation based on the analysis.


In [21]:
from IPython.display  import display, Markdown
display(Markdown(str(result)))

The tool successfully provided a detailed and well-structured report on the best property found for the client, considering prices, trends, and financing options.

Final Answer:
The report includes an in-depth analysis of the properties in São Paulo and Porto Alegre, comparison of market trends, financing options, and a recommendation based on the analysis.